# TFTExplainer variable importance — iteration 4

Run from the **same folder** as notebooks 01/02 (the one containing `darts_logs/`, `series_cache/`, `column_roles.json`, `run_metadata.json`). Nothing from notebook 02 needs to be in memory.

**Run the cells in order.** Cell 1 defines the stubs the checkpoint needs to unpickle.

- Cell 1: imports and callback stubs
- Cell 2: rebuild `val_seq` / `val_cov` from the saved cache (check the printed probe line: 473 days ending 2026-09-30, covariates ending 2026-12-31)
- Cell 3: load the best checkpoint, run TFTExplainer, print tables, save CSVs and plot, stability check

Importances are Variable Selection Network attention weights, not effect on accuracy.

## Cell 1 — imports and stubs

In [ ]:
# ----- CELL 1: imports + stubs needed to unpickle the checkpoint ---------------------
import os, sys, re, json, collections.abc
sys.path.append(os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import TFTModel
from darts.explainability import TFTExplainer
from pytorch_lightning import Callback

from tft_checks import EmbeddingGate          # noqa: F401  (pickled inside the checkpoint)


class TBDiagnostics(Callback):                # stub: the checkpoint pickles __main__.TBDiagnostics
    pass

## Cell 2 — rebuild validation series and covariates

In [ ]:
# ----- CELL 2: rebuild val_seq / val_cov from the files notebook 02 saved -------------
BASE = os.getcwd()
CACHE_DIR = os.path.join(BASE, "series_cache")
with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)
with open(os.path.join(BASE, "run_metadata.json")) as f:
    MODEL_NAME = json.load(f)["model_name"]

target_col        = ROLES["target_col"]
FREQ              = ROLES["freq"]
static_covariates = ROLES["static_covariates"]
binary_covariates = ROLES["binary_covariates"]
series_covariates = ROLES["series_covariates"]
future_covariates = ROLES["future_covariates"]
CALENDAR_COLS     = ROLES["calendar_covariates"]
PETROL_COLS       = ROLES["shared_continuous_covariates"]

# same festive-window list as notebook 02, Section 0
PENALTY_COLS = ([f"N-{i}" for i in range(16, 0, -1)] + ["N"] + [f"N+{i}" for i in range(1, 11)]
                + [f"D-{i}" for i in range(3, 0, -1)] + ["D"] + [f"D+{i}" for i in range(1, 7)]
                + ["C"] + [f"C+{i}" for i in range(1, 7)])
PENALTY_COLS = [c for c in PENALTY_COLS if c in future_covariates]
OTHER_FLAGS  = [c for c in binary_covariates if c not in PENALTY_COLS]


def safe_name(key):
    return str(key).replace("<>", "_").replace("/", "_").replace("\\", "_")


series_index = pd.read_parquet(os.path.join(CACHE_DIR, "series_index.parquet"))
SPEND_PATHS  = np.load(os.path.join(CACHE_DIR, "spend_paths.npy"))
SHARED_COV   = TimeSeries.from_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
CAL_INDEX, CAL_VALUES = SHARED_COV.time_index, SHARED_COV.values(copy=False)
assert len(CAL_INDEX) == SPEND_PATHS.shape[1]


class TargetSeq(collections.abc.Sequence):
    def __init__(self, keys, code_rows):
        self.keys, self.code_rows, self._ram = list(keys), code_rows, {}

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        if i not in self._ram:
            with np.load(os.path.join(CACHE_DIR, f"{safe_name(self.keys[i])}.npz")) as z:
                sales, start = z["val_sales"], str(z["val_start"])
            self._ram[i] = TimeSeries.from_times_and_values(
                pd.date_range(start, periods=len(sales), freq=FREQ),
                sales.reshape(-1, 1).astype(np.float32), columns=[target_col],
                static_covariates=pd.DataFrame([self.code_rows[i]], columns=static_covariates,
                                               dtype=np.float32))
        return self._ram[i]


class SeriesCov(collections.abc.Sequence):
    def __init__(self, path_ids):
        self.path_ids, self._ram = np.asarray(path_ids, dtype=np.int64), {}

    def __len__(self):
        return len(self.path_ids)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        pid = int(self.path_ids[i])
        if pid not in self._ram:
            vals = np.concatenate([CAL_VALUES, SPEND_PATHS[pid]], axis=1).astype(np.float32, copy=False)
            self._ram[pid] = TimeSeries.from_times_and_values(CAL_INDEX, vals, columns=list(future_covariates))
        return self._ram[pid]


# identical selection and order to notebook 02: in_val series, shuffled with seed 42
sel = series_index[series_index["in_val"]].sample(frac=1.0, random_state=42)
val_seq = TargetSeq(sel["key"], sel[static_covariates].to_numpy())
val_cov = SeriesCov(sel["spend_path"].to_numpy())
print(f"val_seq: {len(val_seq):,} series | probe: {val_seq[0].start_time().date()} -> "
      f"{val_seq[0].end_time().date()} ({len(val_seq[0])} days) | cov ends {val_cov[0].end_time().date()}")

## Cell 3 — explain, tables, plots, stability

In [ ]:
# ----- CELL 3: load model + explain ---------------------------------------------------
N_EXPLAIN     = 1000
RUN_STABILITY = True
TOP_K_PLOT    = 25

best = TFTModel.load_from_checkpoint(MODEL_NAME, best=True)      # needs cwd = folder with darts_logs/


def clean_name(name):
    name = str(name)
    m = re.fullmatch(r"static_covariate_(\d+)", name)
    if m:
        return static_covariates[int(m.group(1))]
    return re.sub(r"_(fut_cov|past_cov|static_cov|target|futcov|pastcov)$", "", name)


def group_of(part, var):
    if part == "static":
        return "static (embedded)"
    if var == target_col:
        return "past sales (target history)"
    if "relative_index" in var:
        return "relative index"
    if var in series_covariates:
        return "spend (per-series)"
    if var in PETROL_COLS:
        return "petrol"
    if var in CALENDAR_COLS:
        return "calendar (dow, month-end)"
    if var in PENALTY_COLS:
        return "festive-window flags"
    if var in OTHER_FLAGS:
        return "other binary flags"
    return "UNMATCHED"


def explain_sample(lo, hi):
    ser = [val_seq[i] for i in range(lo, hi)]
    cov = [val_cov[i] for i in range(lo, hi)]
    assert all(c.end_time() >= s.end_time() + pd.Timedelta(days=best.output_chunk_length)
               for s, c in zip(ser[:20], cov[:20])), "future covariates do not reach FORECAST_END"
    explainer = TFTExplainer(best, background_series=ser, background_future_covariates=cov)
    result = explainer.explain()
    try:
        w = result.get_feature_importances()
    except AttributeError:
        w = dict(encoder_importance=result.get_encoder_importance(),
                 decoder_importance=result.get_decoder_importance(),
                 static_covariates_importance=result.get_static_covariates_importance())
    rows = []
    for key, part in [("encoder_importance", "encoder"), ("decoder_importance", "decoder"),
                      ("static_covariates_importance", "static")]:
        df = w.get(key)
        if df is None or len(df) == 0:
            continue
        s = df.mean(axis=0)
        s = s * (100.0 / s.sum())
        for name, val in s.items():
            v = clean_name(name)
            rows.append(dict(part=part, variable=v, importance_pct=float(val), group=group_of(part, v)))
    out = pd.DataFrame(rows)
    out["uniform_pct"] = 100.0 / out.groupby("part")["variable"].transform("count")
    out["x_uniform"] = out["importance_pct"] / out["uniform_pct"]
    return out, result, explainer


imp, result, explainer = explain_sample(0, N_EXPLAIN)
imp = imp.sort_values(["part", "importance_pct"], ascending=[True, False]).reset_index(drop=True)

bad = imp[imp["group"] == "UNMATCHED"]
if len(bad):
    print("*** UNMATCHED variable names (fix clean_name / group_of):", bad["variable"].tolist()[:10])
print(imp.groupby("part").size().to_dict(), "variables per part")
print(imp.groupby("part")["importance_pct"].sum().round(2).to_dict(), "(sum per part)")

wide = (imp.pivot_table(index=["group", "variable"], columns="part",
                        values="importance_pct", aggfunc="first").reset_index())
print("\n=== ALL variables (% of VSN weight within each part) ===")
print(wide.sort_values(["group", "decoder"], ascending=[True, False]).round(3).to_string(index=False))

grp = imp.groupby(["part", "group"])["importance_pct"].sum().unstack("part").fillna(0).round(2)
n_vars = imp.groupby(["part", "group"]).size().unstack("part").fillna(0).astype(int).add_prefix("n_vars_")
print("\n=== By group ===")
print(grp.join(n_vars).to_string())

imp.to_csv(os.path.join(BASE, "tft_variable_importance_long.csv"), index=False)
wide.to_csv(os.path.join(BASE, "tft_variable_importance_wide.csv"), index=False)

parts = [p for p in ["static", "encoder", "decoder"] if p in imp["part"].unique()]
fig, axes = plt.subplots(1, len(parts), figsize=(7 * len(parts), 8))
axes = np.atleast_1d(axes)
for ax, p in zip(axes, parts):
    g = imp[imp["part"] == p].nlargest(TOP_K_PLOT, "importance_pct").sort_values("importance_pct")
    ax.barh(g["variable"], g["importance_pct"])
    ax.axvline(g["uniform_pct"].iloc[0], ls="--", c="grey", lw=1)
    ax.set_title(f"{p}: top {min(TOP_K_PLOT, len(g))}  (dashed = equal weights)")
    ax.set_xlabel("% of VSN weight")
plt.tight_layout()
plt.savefig(os.path.join(BASE, "tft_variable_importance.png"), dpi=150)
plt.show()

if RUN_STABILITY:
    imp_b, _, _ = explain_sample(N_EXPLAIN, 2 * N_EXPLAIN)
    for p in parts:
        a = imp[imp.part == p].set_index("variable")["importance_pct"]
        b = imp_b[imp_b.part == p].set_index("variable")["importance_pct"].reindex(a.index)
        print(f"rank agreement (Spearman), two disjoint samples, {p:8s}: {a.corr(b, method='spearman'):.3f}")